# CondorBrain Training with Predicate Discovery

This notebook trains a neural model that **jointly learns**:
1. **Which inequality patterns matter** (e.g., `H[0] > H[1] AND L[0] > L[1]`)
2. **How to combine them with raw features** for trading decisions

## Key Concepts

**Predicates** are human-readable inequality patterns like:
- `close[0] > close[5]` (price momentum)
- `{high[0] - low[0]} > {high[1] - low[1]}` (range expansion)
- `rsi_dyn[0] < 30 AND close[0] > bb_lower_dyn[0]` (oversold bounce)

The model searches through **millions of possible combinations** and finds the ones that predict well.


## 1. Setup

In [ ]:
# Install dependencies (run once)
!pip install pandas numpy torch --quiet

In [ ]:
# Upload your data file
from google.colab import files
print("Upload your data CSV (mamba_institutional_2024_1m_v22.csv)")
uploaded = files.upload()
data_file = list(uploaded.keys())[0]
print(f"Uploaded: {data_file}")

## 2. Core Implementation (Self-Contained)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from typing import List, Tuple, Dict, Any
from dataclasses import dataclass
from enum import IntEnum

# V2.2 Feature Schema
FEATURE_COLS_V22 = [
    "open", "high", "low", "close", "volume",
    "delta", "gamma", "vega", "theta", "iv", "ivr", "spread_ratio", "te", "strike",
    "target_spot", "max_dd_60m",
    "log_return", "vol_ewma", "ret_z", "atr_pct", "kappa_proxy", "vol_energy",
    "rsi_dyn", "adx_adaptive", "psar_adaptive",
    "bb_mu_dyn", "bb_sigma_dyn", "bb_lower_dyn", "bb_upper_dyn",
    "stoch_k_dyn", "consolidation_score", "breakout_score",
    "bb_percentile", "bw_expansion_rate", "cmf", "pressure_up", "pressure_down",
    "friction_ratio", "exec_allow", "gap_risk_score", "risk_override", "iv_confidence",
    "mtf_consensus", "macd_norm", "macd_signal_norm", "macd_histogram",
    "plus_di", "minus_di", "psar_trend", "psar_reversion_mu",
    "beta1_norm_stub", "chaos_membership", "position_size_mult", "fuzzy_reversion_11",
]

ALL_FIELDS = FEATURE_COLS_V22.copy()
FIELD_TO_IDX = {f: i for i, f in enumerate(ALL_FIELDS)}
IDX_TO_FIELD = {i: f for i, f in enumerate(ALL_FIELDS)}
N_FIELDS = len(ALL_FIELDS)

print(f"V2.2 Schema: {N_FIELDS} fields")

In [ ]:
# Grammar Definition

class ArithOp(IntEnum):
    NONE = 0; ADD = 1; SUB = 2; MUL = 3; DIV = 4

class CompareOp(IntEnum):
    GT = 0; LT = 1; GTE = 2; LTE = 3; EQ = 4

class LogicOp(IntEnum):
    AND = 0; OR = 1

COMPARE_SYMBOLS = ['>', '<', '>=', '<=', '==']
ARITH_SYMBOLS = ['', '+', '-', '*', '/']
LOGIC_SYMBOLS = ['AND', 'OR']

@dataclass
class Atom:
    field1: int; lookback1: int; arith_op: ArithOp
    field2: int = 0; lookback2: int = 0
    
    def is_simple(self): return self.arith_op == ArithOp.NONE
    
    def __str__(self):
        f1 = IDX_TO_FIELD.get(self.field1, f"F{self.field1}")
        if self.is_simple(): return f"{f1}[{self.lookback1}]"
        f2 = IDX_TO_FIELD.get(self.field2, f"F{self.field2}")
        return f"{{{f1}[{self.lookback1}]{ARITH_SYMBOLS[self.arith_op]}{f2}[{self.lookback2}]}}"

@dataclass
class Inequality:
    left: Atom; compare: CompareOp; right: Atom
    def __str__(self): return f"{self.left} {COMPARE_SYMBOLS[self.compare]} {self.right}"

@dataclass
class Predicate:
    inequalities: List[Inequality]; logic_ops: List[LogicOp]
    def __str__(self):
        if len(self.inequalities) == 1: return str(self.inequalities[0])
        parts = [str(self.inequalities[0])]
        for ineq, op in zip(self.inequalities[1:], self.logic_ops):
            parts.append(f" {LOGIC_SYMBOLS[op]} {ineq}")
        return ''.join(parts)

print("Grammar loaded")

In [ ]:
# Predicate Selector (Differentiable)

class PredicateSelector(nn.Module):
    def __init__(self, n_slots=2048, max_active=256, d_embed=128, n_fields=N_FIELDS, max_lookback=128, temp=1.0):
        super().__init__()
        self.n_slots, self.max_active, self.n_fields = n_slots, max_active, n_fields
        self.max_lookback, self.temperature = max_lookback, temp
        
        self.predicate_embeddings = nn.Parameter(torch.randn(n_slots, d_embed) * 0.02)
        self.decoder = nn.Sequential(
            nn.Linear(d_embed, d_embed * 2), nn.GELU(), nn.LayerNorm(d_embed * 2),
            nn.Linear(d_embed * 2, d_embed * 2), nn.GELU(),
        )
        
        # Parameter heads
        self.left_field1 = nn.Linear(d_embed * 2, n_fields)
        self.left_lookback1 = nn.Linear(d_embed * 2, max_lookback + 1)
        self.left_arith = nn.Linear(d_embed * 2, 5)
        self.left_field2 = nn.Linear(d_embed * 2, n_fields)
        self.left_lookback2 = nn.Linear(d_embed * 2, max_lookback + 1)
        self.compare_op = nn.Linear(d_embed * 2, 5)
        self.right_field1 = nn.Linear(d_embed * 2, n_fields)
        self.right_lookback1 = nn.Linear(d_embed * 2, max_lookback + 1)
        self.right_arith = nn.Linear(d_embed * 2, 5)
        self.right_field2 = nn.Linear(d_embed * 2, n_fields)
        self.right_lookback2 = nn.Linear(d_embed * 2, max_lookback + 1)
        
        self.importance_logits = nn.Parameter(torch.zeros(n_slots))
    
    def _gumbel_sample(self, logits):
        if self.training:
            probs = F.gumbel_softmax(logits, tau=self.temperature, hard=True)
            return (probs * torch.arange(logits.size(-1), device=logits.device, dtype=logits.dtype)).sum(-1)
        return logits.argmax(-1).float()
    
    def forward(self, return_params=False):
        importance = torch.sigmoid(self.importance_logits)
        if not return_params: return importance, None
        
        hidden = self.decoder(self.predicate_embeddings)
        params = torch.stack([
            self._gumbel_sample(self.left_field1(hidden)),
            self._gumbel_sample(self.left_lookback1(hidden)),
            self._gumbel_sample(self.left_arith(hidden)),
            self._gumbel_sample(self.left_field2(hidden)),
            self._gumbel_sample(self.left_lookback2(hidden)),
            self._gumbel_sample(self.compare_op(hidden)),
            self._gumbel_sample(self.right_field1(hidden)),
            self._gumbel_sample(self.right_lookback1(hidden)),
            self._gumbel_sample(self.right_arith(hidden)),
            self._gumbel_sample(self.right_field2(hidden)),
            self._gumbel_sample(self.right_lookback2(hidden)),
        ], dim=-1)
        return importance, params
    
    def sparsity_loss(self): return torch.sigmoid(self.importance_logits).sum()
    
    def get_active_predicates(self, threshold=0.1, max_return=None):
        self.eval()
        with torch.no_grad():
            importance, params = self.forward(return_params=True)
            mask = importance > threshold
            active_idx = torch.where(mask)[0]
            if len(active_idx) == 0: return [], np.array([]), []
            
            sort_idx = torch.argsort(importance[active_idx], descending=True)
            active_idx = active_idx[sort_idx]
            if max_return: active_idx = active_idx[:max_return]
            
            active_params = params[active_idx].cpu().numpy().astype(int)
            active_importance = importance[active_idx].cpu().numpy()
            
            predicates, names = [], []
            for i, p in enumerate(active_params):
                left = Atom(p[0], p[1], ArithOp(p[2]), p[3], p[4])
                right = Atom(p[6], p[7], ArithOp(p[8]), p[9], p[10])
                pred = Predicate([Inequality(left, CompareOp(p[5]), right)], [])
                predicates.append(pred)
                names.append(f"{pred} (imp={active_importance[i]:.3f})")
            return predicates, active_importance, names

print("PredicateSelector loaded")

In [ ]:
# Predicate Evaluator (Vectorized GPU)

def evaluate_predicates_gpu(data, params, importance, max_active=256, eps=1e-8):
    if data.dim() == 2: data = data.unsqueeze(0); squeeze_out = True
    else: squeeze_out = False
    
    batch, seq_len, n_fields = data.shape
    device = data.device
    K = params.shape[0]
    
    top_k = min(max_active, K)
    _, top_idx = torch.topk(importance, top_k)
    active_params = params[top_idx]
    active_importance = importance[top_idx]
    
    # Extract parameters
    l_f1, l_n1, l_op = active_params[:, 0].long(), active_params[:, 1].long(), active_params[:, 2].long()
    l_f2, l_n2 = active_params[:, 3].long(), active_params[:, 4].long()
    cmp = active_params[:, 5].long()
    r_f1, r_n1, r_op = active_params[:, 6].long(), active_params[:, 7].long(), active_params[:, 8].long()
    r_f2, r_n2 = active_params[:, 9].long(), active_params[:, 10].long()
    
    max_lb = min(torch.max(torch.stack([l_n1, l_n2, r_n1, r_n2])).item(), seq_len - 1)
    result = torch.zeros(batch, seq_len, max_active, device=device, dtype=data.dtype)
    
    for t in range(max_lb, seq_len):
        left_val = torch.zeros(batch, top_k, device=device, dtype=data.dtype)
        right_val = torch.zeros(batch, top_k, device=device, dtype=data.dtype)
        
        for k in range(top_k):
            t_l1 = max(0, t - l_n1[k].item())
            left_val[:, k] = data[:, t_l1, l_f1[k]]
            if l_op[k] > 0:
                t_l2 = max(0, t - l_n2[k].item())
                val2 = data[:, t_l2, l_f2[k]]
                if l_op[k] == 1: left_val[:, k] += val2
                elif l_op[k] == 2: left_val[:, k] -= val2
                elif l_op[k] == 3: left_val[:, k] *= val2
                elif l_op[k] == 4: left_val[:, k] /= (val2 + eps)
            
            t_r1 = max(0, t - r_n1[k].item())
            right_val[:, k] = data[:, t_r1, r_f1[k]]
            if r_op[k] > 0:
                t_r2 = max(0, t - r_n2[k].item())
                val2 = data[:, t_r2, r_f2[k]]
                if r_op[k] == 1: right_val[:, k] += val2
                elif r_op[k] == 2: right_val[:, k] -= val2
                elif r_op[k] == 3: right_val[:, k] *= val2
                elif r_op[k] == 4: right_val[:, k] /= (val2 + eps)
        
        diff = left_val - right_val
        steep = 10.0
        for k in range(top_k):
            if cmp[k] in [0, 2]: result[:, t, k] = torch.sigmoid(steep * diff[:, k])
            elif cmp[k] in [1, 3]: result[:, t, k] = torch.sigmoid(-steep * diff[:, k])
            elif cmp[k] == 4: result[:, t, k] = torch.exp(-steep * diff[:, k].abs())
    
    result[:, :, :top_k] *= active_importance.unsqueeze(0).unsqueeze(0)
    if squeeze_out: result = result.squeeze(0)
    return result

print("Evaluator loaded")

In [ ]:
# Predicate-Augmented Model

class PredicateAugmentedModel(nn.Module):
    def __init__(self, input_dim=54, d_model=256, n_layers=4, n_heads=8,
                 n_predicate_slots=2048, max_active=256, n_outputs=10, dropout=0.1):
        super().__init__()
        self.input_dim, self.d_model, self.max_active = input_dim, d_model, max_active
        
        self.predicate_selector = PredicateSelector(n_predicate_slots, max_active, n_fields=input_dim)
        
        combined_dim = input_dim + max_active
        self.input_proj = nn.Sequential(
            nn.Linear(combined_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout)
        )
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, dropout, batch_first=True, norm_first=True
        )
        self.backbone = nn.TransformerEncoder(encoder_layer, n_layers)
        
        self.output_heads = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.LayerNorm(d_model), nn.Linear(d_model, n_outputs)
        )
    
    def forward(self, x):
        importance, params = self.predicate_selector(return_params=True)
        pred_features = evaluate_predicates_gpu(x, params, importance, self.max_active)
        combined = torch.cat([x, pred_features], dim=-1)
        h = self.input_proj(combined)
        h = self.backbone(h)
        return self.output_heads(h[:, -1, :])
    
    def compute_loss(self, pred, target, sparsity_weight=0.001):
        pred_loss = F.mse_loss(pred, target)
        sparse_loss = self.predicate_selector.sparsity_loss()
        total = pred_loss + sparsity_weight * sparse_loss
        with torch.no_grad():
            importance, _ = self.predicate_selector(return_params=False)
            n_active = (importance > 0.1).sum().item()
        return total, {'pred': pred_loss.item(), 'sparse': sparse_loss.item(), 'total': total.item(), 'n_active': n_active}
    
    def get_discovered_predicates(self):
        _, _, names = self.predicate_selector.get_active_predicates(threshold=0.1)
        return names

print("Model loaded")

## 3. Data Preparation

In [ ]:
# Load and prepare data
print(f"Loading {data_file}...")
df = pd.read_csv(data_file)
print(f"Loaded {len(df):,} rows")

# Find available features
available = [c for c in FEATURE_COLS_V22 if c in df.columns]
print(f"Using {len(available)}/{len(FEATURE_COLS_V22)} features")

# Extract and normalize
X = df[available].fillna(0).values.astype(np.float32)
med = np.median(X, axis=0)
mad = np.median(np.abs(X - med), axis=0)
scale = np.where(mad < 1e-8, 1.0, 1.4826 * mad)
X = np.clip((X - med) / scale, -10, 10).astype(np.float32)

# Generate targets (forward returns)
close = df['close'].values if 'close' in df.columns else df.iloc[:, 3].values
ret = np.zeros(len(df))
ret[:-5] = (close[5:] - close[:-5]) / (close[:-5] + 1e-8)

y = np.column_stack([
    np.zeros(len(df)), np.zeros(len(df)),
    np.full(len(df), 5.0), np.full(len(df), 7.0),
    np.full(len(df), 0.7), ret,
    np.full(len(df), 0.5), np.full(len(df), 0.5),
    (ret > 0).astype(float), (ret < 0).astype(float),
]).astype(np.float32)
y = np.nan_to_num(np.clip(y, -10, 10), 0)

print(f"X shape: {X.shape}, y shape: {y.shape}")

## 4. Training

In [ ]:
# Configuration
CONFIG = {
    'd_model': 256,
    'n_layers': 4,
    'n_heads': 8,
    'predicate_slots': 2048,
    'max_active': 256,
    'lookback': 128,
    'batch_size': 32,
    'epochs': 10,
    'lr': 1e-4,
    'sparsity_weight': 0.001,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    use_bf16 = torch.cuda.is_bf16_supported()
else:
    use_bf16 = False

In [ ]:
# Prepare tensors
L = CONFIG['lookback']
B = CONFIG['batch_size']

# Split
split = int(len(X) * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# To GPU
dtype_gpu = torch.bfloat16 if use_bf16 else torch.float16
X_train_t = torch.from_numpy(X_train).to(device, dtype_gpu)
y_train_t = torch.from_numpy(y_train).to(device, torch.float32)
X_val_t = torch.from_numpy(X_val).to(device, dtype_gpu)
y_val_t = torch.from_numpy(y_val).to(device, torch.float32)

# Create sequences
X_train_seq = X_train_t.unfold(0, L, 1).permute(0, 2, 1)
X_val_seq = X_val_t.unfold(0, L, 1).permute(0, 2, 1)

n_train = X_train_seq.shape[0]
n_val = X_val_seq.shape[0]
n_train_batches = n_train // B
n_val_batches = max(1, n_val // B)

print(f"Train: {n_train:,} sequences, {n_train_batches} batches")
print(f"Val: {n_val:,} sequences, {n_val_batches} batches")

In [ ]:
# Create model
model = PredicateAugmentedModel(
    input_dim=X_train_t.shape[1],
    d_model=CONFIG['d_model'],
    n_layers=CONFIG['n_layers'],
    n_heads=CONFIG['n_heads'],
    n_predicate_slots=CONFIG['predicate_slots'],
    max_active=CONFIG['max_active'],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, CONFIG['epochs'])
scaler = GradScaler('cuda') if device.type == 'cuda' else None

In [ ]:
# Training loop
history = {'train': [], 'val': [], 'n_preds': []}
best_val = float('inf')
best_predicates = []

print(f"\n{'='*60}")
print("TRAINING WITH PREDICATE DISCOVERY")
print(f"{'='*60}\n")

for epoch in range(CONFIG['epochs']):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss, n_active = 0, 0
    perm = torch.randperm(n_train_batches)
    
    for bi in range(n_train_batches):
        idx = perm[bi].item()
        s, e = idx * B, idx * B + B
        batch_x = X_train_seq[s:e]
        batch_y = y_train_t[s + L:e + L]
        
        optimizer.zero_grad()
        if scaler:
            with autocast('cuda', dtype=torch.bfloat16 if use_bf16 else torch.float16):
                pred = model(batch_x)
                loss, metrics = model.compute_loss(pred, batch_y, CONFIG['sparsity_weight'])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(batch_x)
            loss, metrics = model.compute_loss(pred, batch_y, CONFIG['sparsity_weight'])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        train_loss += metrics['total']
        n_active = metrics['n_active']
    
    train_loss /= n_train_batches
    
    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for bi in range(n_val_batches):
            s, e = bi * B, min(bi * B + B, n_val)
            batch_x = X_val_seq[s:e]
            batch_y = y_val_t[s + L:e + L]
            with autocast('cuda', dtype=torch.bfloat16 if use_bf16 and scaler else torch.float32):
                pred = model(batch_x)
                val_loss += F.mse_loss(pred, batch_y).item()
    val_loss /= n_val_batches
    
    scheduler.step()
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['n_preds'].append(n_active)
    
    predicates = model.get_discovered_predicates()
    
    print(f"Epoch {epoch+1:2d}/{CONFIG['epochs']} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
          f"Predicates: {n_active:3d} | Time: {time.time()-t0:.1f}s")
    
    if val_loss < best_val:
        best_val = val_loss
        best_predicates = predicates
        print(f"  --> New best! Sample predicates: {predicates[:2] if predicates else 'none'}")

## 5. Results

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train'], label='Train')
axes[0].plot(history['val'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(history['n_preds'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Active Predicates')
axes[1].set_title('Discovered Predicates')

plt.tight_layout()
plt.show()

In [ ]:
# Show discovered predicates
print(f"\n{'='*60}")
print("DISCOVERED PREDICATES")
print(f"{'='*60}\n")

predicates, importance, names = model.predicate_selector.get_active_predicates(threshold=0.1)
print(f"Total active predicates: {len(predicates)}")

print("\nTop 30 Predicates by Importance:")
print("-" * 60)
for i, name in enumerate(names[:30]):
    print(f"{i+1:2d}. {name}")

In [ ]:
# Export predicates
export = {
    'n_predicates': len(predicates),
    'best_val_loss': best_val,
    'predicates': [
        {'expression': str(p), 'importance': float(imp)}
        for p, imp in zip(predicates, importance)
    ]
}

with open('discovered_predicates.json', 'w') as f:
    json.dump(export, f, indent=2)

print(f"\nExported {len(predicates)} predicates to discovered_predicates.json")

# Download
files.download('discovered_predicates.json')

In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'discovered_predicates': names,
    'best_val_loss': best_val,
    'normalization': {'median': med.tolist(), 'scale': scale.tolist()},
}, 'condor_with_predicates.pth')

print("Model saved to condor_with_predicates.pth")
files.download('condor_with_predicates.pth')